# 25 — Umpires and challenges (2026)

For each plate umpire: how many challenges were made against their calls,
how many were overturned, and the overturn rate with a 95% interval.
Minimum 10 plate games.


In [ ]:
import sys

import pandas as pd

sys.path.insert(0, "../src")
from stats_helpers import rate_table

TABLES = "../data/tables"
REPORTS = "../reports/tables"

ch = pd.read_parquet(f"{TABLES}/challenges/all_seasons.parquet")
games = pd.read_parquet(f"{TABLES}/games/all_seasons.parquet")
rank = pd.read_csv(f"{REPORTS}/umpire_rankings.csv")

print(ch.columns.tolist())
print(ch.shape)

In [ ]:
umps = games[["game_pk", "plate_umpire_id", "plate_umpire_name"]]
ch = ch.merge(umps, on="game_pk", how="left")

per_ump = rate_table(
    ch, ["plate_umpire_id", "plate_umpire_name"], "is_overturned"
).rename(
    columns={
        "plate_umpire_id": "official_id",
        "plate_umpire_name": "official_name",
        "trials": "challenges",
        "successes": "overturned",
        "rate": "overturn_rate",
    }
)

r26 = rank[rank["season"] == 2026][
    ["official_id", "plate_games", "called_pitches", "accuracy"]
]
per_ump = per_ump.merge(r26, on="official_id", how="inner")
per_ump["challenges_per_game"] = per_ump["challenges"] / per_ump["plate_games"]
per_ump["overturns_per_game"] = per_ump["overturned"] / per_ump["plate_games"]
per_ump = per_ump.sort_values("overturn_rate")

per_ump.to_csv(f"{REPORTS}/umpire_challenges_2026.csv", index=False)
print(len(per_ump), "umpires (10+ plate games)")
print(per_ump.round(3).head(10).to_string(index=False))
print()
print(per_ump.round(3).tail(10).to_string(index=False))
print()
print("correlation accuracy vs overturn rate:",
      round(per_ump["accuracy"].corr(per_ump["overturn_rate"]), 3))

## Takeaways

- Each umpire faced roughly 70–150 challenges in 2026, about 3–5 per plate game.
- Overturn rate = the share of challenges where the umpire turned out wrong.
  Lowest: Visconti 42%, Ripperger 44% (the top of the accuracy ranking).
  Highest: Andy Fletcher 66%, Rob Drake 65%, Adrian Johnson 64%. League 53.8%.
- Accuracy and overturn rate move together: correlation −0.72 across 89 umpires.
  The less accurate the umpire, the more challenges go against them. Jorge's J5
  does the full version with intervals and experience.
- Dillon Wilson (36%) has only 31 challenges — interval 21–53%, so not a real numbers yet.